# Notebook 02 — First API call + strict-schema retry + the cacheable analyze step

**Purpose:** First real ingest. Implement the §7.3.1 retry pattern end-to-end on a single source. Make the *analyze* step explicitly cacheable so NB 10 has something to cache.

- Confirms first API call succeeds (smoke test).
- Implements the §7.3.1 retry pattern end-to-end on a single source.
- Splits ingest into a cacheable analyze step and a state-dependent synthesize step.
- Demonstrates draft-fallback after 3 failures.

In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

In [ ]:
# Import shared notebook dependencies for schema validation and frontmatter parsing.
from pathlib import Path
from datetime import date
from enum import Enum
from typing import Literal, Annotated

from pydantic import BaseModel, Field, ValidationError, field_validator, TypeAdapter
import frontmatter
import json

import os
from dotenv import load_dotenv
from anthropic import Anthropic           # for the smoke test
# Claude Agent SDK used for the agent path; the smoke test can use the raw SDK
from claude_agent_sdk import query, ClaudeAgentOptions


# Load page schema models from the engine package.
from engine.models.pages import Page, EntityPage, ConceptPage, SourcePage, DecisionPage, MeetingPage, MetricPage, QAPage, AnalysisPage      # all 8
from engine.models.wiki_config import MarginaliaConfig


In [ ]:
# assert API key presence for the ingest smoke test.

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

In [ ]:
client = Anthropic()
resp = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=64,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
)
print(resp.content[0].text, resp.usage)

In [ ]:
# Load purpose.md / AGENTS.md
config = MarginaliaConfig.load(Path("data/poc-wiki"))
print(config.purpose_body[:200], config.agents_body[:200])

In [ ]:
from engine.agents.ingest.analyze import SourceAnalysis 


In [ ]:
# Draft ingest analyze prompt v1 (system + user template) following §7.3 patterns.
PROMPT_VERSION = "v1"

ANALYZE_SYSTEM = """You are the Marginalia ingest analyzer.

You transform one source document into structured analysis JSON.
This step must stay cacheable and deterministic for a given source content and prompt version.

<rules>
- Return only a single JSON object that matches the provided schema exactly.
- No markdown fences, no prose preamble, no trailing commentary.
- Do not output <thinking> or chain-of-thought.
- Be faithful to source evidence; do not invent facts.
- Keep entities as bare names (no wikilinks in analyze step).
</rules>

<examples>
<example>
<input kind="granola_meeting">
Q2 recap:
- Sandro reviewed Apollo launch blockers with Priya and Marco.
- Decision: move release to 2026-05-12.
- Follow-up owner: Priya.
</input>
<output>
{
  "proposed_title": "Q2 Apollo Launch Blockers Recap",
  "proposed_type": "meeting",
  "summary": "Meeting recap covering Apollo launch blockers, a release date move to 2026-05-12, and follow-up ownership assigned to Priya.",
  "entities": ["Sandro", "Priya", "Marco", "Apollo"],
  "proposed_tags": ["meeting", "release", "apollo"],
  "source_kind": "granola_meeting",
  "confidence": "high",
  "content_sha256": "0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef"
}
</output>
</example>

<example>
<input kind="slack_thread">
SP posted: "Quick note: Sandro a.k.a. SP will own migration sign-off."
Follow-up from Elena: "SP confirmed rollback plan is ready."
</input>
<output>
{
  "proposed_title": "Migration Sign-off Ownership Thread",
  "proposed_type": "decision",
  "summary": "Slack discussion confirms Sandro (also referred to as SP) owns migration sign-off and that rollback planning is complete.",
  "entities": ["Sandro", "SP", "Elena"],
  "proposed_tags": ["migration", "ownership", "decision"],
  "source_kind": "slack_thread",
  "confidence": "medium",
  "content_sha256": "fedcba9876543210fedcba9876543210fedcba9876543210fedcba9876543210"
}
</output>
</example>

<example>
<input kind="notion_page">
The Apollo team asked for a KPI refresh. Apollo remains a key initiative this quarter.
</input>
<output>
{
  "proposed_title": "Apollo KPI Refresh Request",
  "proposed_type": "analysis",
  "summary": "Document requests a KPI refresh for the Apollo team and notes Apollo as a priority initiative this quarter, without resolving whether the references denote the same entity.",
  "entities": ["Apollo team", "Apollo"],
  "proposed_tags": ["apollo", "kpi", "analysis"],
  "source_kind": "notion_page",
  "confidence": "low",
  "content_sha256": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa"
}
</output>
</example>
</examples>
"""
# Append config.purpose_body and config.agents_body to the system prompt at call time.

ANALYZE_USER_TEMPLATE = """\
<source kind="{source_kind}" sha256="{sha}">
{content}
</source>

<schema>
{schema_json}
</schema>

<instructions>
Extract the structured analysis. Return ONLY a JSON object matching the schema.
Do not include thinking, prose, or markdown fences.
</instructions>
"""

schema_json = json.dumps(SourceAnalysis.model_json_schema())

In [ ]:
# Cacheable analyze step: pure function over source content + prompt version.
import hashlib

from engine.models.pages import SourceKind


def _parse_json_object(raw: str) -> dict:
    """Extract and parse the first JSON object from model text output."""
    text = raw.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        return json.loads(text[start : end + 1])


async def analyze_source(content: str, source_kind: SourceKind) -> SourceAnalysis:
    sha = hashlib.sha256(content.encode()).hexdigest()
    user_msg = ANALYZE_USER_TEMPLATE.format(
        source_kind=source_kind.value,
        sha=sha,
        content=content,
        schema_json=json.dumps(SourceAnalysis.model_json_schema()),
    )

    purpose = getattr(config, "purpose", None) or config.purpose_body
    agents_style = getattr(config, "agents_style", None) or config.agents_body
    system = (
        ANALYZE_SYSTEM
        + f"\n\n<wiki_purpose>\n{purpose}\n</wiki_purpose>"
        + f"\n\n<style_guide>\n{agents_style}\n</style_guide>"
    )

    # temperature=0 for cacheability. Caching with sampling is theatre.
    # Use the raw anthropic SDK here, not the Agent SDK; this is a single Messages call, no tools, no loop.
    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        temperature=0,
        system=system,
        messages=[{"role": "user", "content": user_msg}],
    )

    raw = resp.content[0].text
    data = _parse_json_object(raw)  # JSON parse failure surfaces as a retry trigger.
    # We override content_sha256 post-hoc — never let the model pick the cache key.
    data["content_sha256"] = sha
    return SourceAnalysis.model_validate(data)

In [ ]:
# Cacheability sanity check: run analyze twice and compare stable fields.
good_source = Path("data/poc-wiki/sources/good_source.md")
good_source.parent.mkdir(parents=True, exist_ok=True)
if not good_source.exists():
    good_source.write_text(
        """# Apollo launch sync\n\nSandro and Priya reviewed Apollo launch blockers.\nThe team agreed to keep weekly check-ins and track KPI drift.\n""",
        encoding="utf-8",
    )

content = good_source.read_text(encoding="utf-8")
a1 = await analyze_source(content, SourceKind.LOCAL_FILE)
a2 = await analyze_source(content, SourceKind.LOCAL_FILE)

stable_fields_equal = (
    a1.proposed_title == a2.proposed_title
    and a1.proposed_type == a2.proposed_type
    and a1.entities == a2.entities
    and a1.proposed_tags == a2.proposed_tags
)

print("Run 1:", {
    "proposed_title": a1.proposed_title,
    "proposed_type": a1.proposed_type,
    "entities": a1.entities,
    "proposed_tags": a1.proposed_tags,
    "summary": a1.summary,
})
print("Run 2:", {
    "proposed_title": a2.proposed_title,
    "proposed_type": a2.proposed_type,
    "entities": a2.entities,
    "proposed_tags": a2.proposed_tags,
    "summary": a2.summary,
})
print("Stable fields match exactly:", stable_fields_equal)
if not stable_fields_equal:
    print("Prompt has too much wiggle. Tighten extraction instructions before moving on.")

In [ ]:
# Draft ingest synthesize prompt v1 (analysis -> SourcePage).
SYNTH_PROMPT_VERSION = "v1"

SYNTHESIZE_SYSTEM = """You are the Marginalia ingest synthesizer.

Transform analyzed source data into a complete SourcePage.
Follow schema constraints exactly and produce two tagged blocks in the response.

Rules:
- Output exactly one <frontmatter>...</frontmatter> block and one <body>...</body> block.
- <frontmatter> must contain JSON only, matching the supplied SourcePage schema.
- <body> must contain markdown only.
- Do not include <thinking>, chain-of-thought, or extra prose outside the tags.
- Return SourcePage frontmatter that validates on first attempt.
- Set type to "source".
- Set status to "active" for successful happy-path synthesis.
- Include a non-empty sources list (at least one SourceRef with ref, kind, captured, authority).
- Ensure created and last_synced are ISO dates.
- Do not include owners, related, contradicts, or supersedes unless values are valid wikilinks.
- Prefer leaving wikilink-constrained fields empty when uncertain.
"""

SYNTHESIZE_USER_TEMPLATE = """\
<analysis>
{analysis_json}
</analysis>

<existing_pages>
{existing_pages_json}
</existing_pages>

<schema>
{schema_json}
</schema>

<instructions>
Produce a complete SourcePage.
Output frontmatter as JSON inside <frontmatter>...</frontmatter> and body markdown inside <body>...</body>.
</instructions>
"""

source_page_schema_json = json.dumps(SourcePage.model_json_schema())
existing_pages_json = json.dumps([])

# Why split frontmatter and body?
# Frontmatter is strictly validated via the Page discriminated union;
# body stays freeform markdown without JSON escaping overhead.

In [ ]:
# Synthesize step with strict-schema retry loop (§7.3.1).
import re
import time

from engine.models.pages import PageStatus, PageType

MAX_ATTEMPTS = 3

# Keep compatibility with both naming variants used across notebook drafts.
SYNTH_SYSTEM = SYNTHESIZE_SYSTEM
SYNTH_USER_TEMPLATE = SYNTHESIZE_USER_TEMPLATE

purpose = getattr(config, "purpose", None) or config.purpose_body
agents_style = getattr(config, "agents_style", None) or config.agents_body
scope_blocks = (
    f"\n\n<wiki_purpose>\n{purpose}\n</wiki_purpose>"
    f"\n\n<style_guide>\n{agents_style}\n</style_guide>"
)


def _parse_frontmatter_and_body(raw: str) -> tuple[dict, str]:
    """Parse <frontmatter> JSON and <body> markdown from model output."""
    fm_match = re.search(r"<frontmatter>\s*(\{.*?\})\s*</frontmatter>", raw, re.DOTALL)
    body_match = re.search(r"<body>\s*(.*?)\s*</body>", raw, re.DOTALL)
    if not fm_match or not body_match:
        raise ValueError("response must include <frontmatter>...</frontmatter> and <body>...</body>")

    fm_dict = json.loads(fm_match.group(1))
    body = body_match.group(1).strip()
    return fm_dict, body


def _normalize_errors(exc: Exception) -> list[dict]:
    """Normalize error payload for feedback into next retry attempt."""
    if isinstance(exc, ValidationError):
        return [
            {
                "loc": list(item.get("loc", [])),
                "msg": item.get("msg", "validation error"),
                "type": item.get("type", "value_error"),
            }
            for item in exc.errors()
        ]

    if isinstance(exc, json.JSONDecodeError):
        return [
            {
                "loc": ["response"],
                "msg": f"invalid JSON: {exc.msg}",
                "type": "json_decode_error",
            }
        ]

    return [{"loc": ["response"], "msg": str(exc), "type": exc.__class__.__name__}]


async def synthesize_page(
    analysis: SourceAnalysis,
    hint: str | None,
    existing_pages: list[dict],
) -> tuple[SourcePage, list[dict]]:
    """Returns (page, attempt_log). page may be a draft with validation_errors."""
    attempt_log: list[dict] = []
    prior_errors: list[dict] = []

    for attempt in range(1, MAX_ATTEMPTS + 1):
        user_msg = SYNTH_USER_TEMPLATE.format(
            analysis_json=analysis.model_dump_json(),
            existing_pages_json=json.dumps(existing_pages),
            schema_json=json.dumps(SourcePage.model_json_schema()),
        )

        if hint:
            user_msg += f"\n\n<hint>\n{hint}\n</hint>"
        if prior_errors:
            user_msg += (
                "\n\n<validation_errors>\n"
                + json.dumps(prior_errors, indent=2)
                + "\n</validation_errors>"
            )

        t0 = time.monotonic()
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2048,
            temperature=0,
            system=SYNTH_SYSTEM + scope_blocks,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = resp.content[0].text
        attempt_log.append(
            {
                "attempt": attempt,
                "ms": int((time.monotonic() - t0) * 1000),
                "input_tokens": resp.usage.input_tokens,
                "output_tokens": resp.usage.output_tokens,
            }
        )

        try:
            fm_dict, body = _parse_frontmatter_and_body(raw)
            _ = body  # body is validated by contract shape; markdown stays freeform.
            page = SourcePage.model_validate(fm_dict)
            return page, attempt_log
        except (ValidationError, ValueError, json.JSONDecodeError) as exc:
            # Feed structured errors back; this is the core §7.3.1 retry pattern.
            prior_errors = _normalize_errors(exc)
            attempt_log[-1]["validation_errors"] = prior_errors

    # 3 failures — land as draft per §7.3.1.
    # model_construct is intentional here: fallback drafts can bypass strict validators.
    draft = SourcePage.model_construct(
        title=analysis.proposed_title or "[unknown]",
        type=PageType.SOURCE,
        status=PageStatus.DRAFT,
        created=date.today(),
        last_synced=date.today(),
        sources=[],
        validation_errors=[json.dumps(err) for err in prior_errors],
    )
    return draft, attempt_log

In [ ]:
# Helper functions for synthesize retry parsing and structured error feedback.
import re


def _parse_frontmatter_and_body(raw: str) -> tuple[dict, str]:
    """Extract <frontmatter> JSON and <body> markdown from model output."""
    fm_match = re.search(r"<frontmatter>\s*(.*?)\s*</frontmatter>", raw, re.DOTALL)
    body_match = re.search(r"<body>\s*(.*?)\s*</body>", raw, re.DOTALL)
    if not fm_match or not body_match:
        raise ValueError("response must include <frontmatter>...</frontmatter> and <body>...</body>")

    try:
        fm_dict = json.loads(fm_match.group(1).strip())
    except json.JSONDecodeError as exc:
        raise ValueError(f"malformed <frontmatter> JSON: {exc.msg}") from exc

    return fm_dict, body_match.group(1).strip()


def _normalize_errors(exc: Exception) -> list[dict]:
    """Return normalized retry payload entries shaped as {loc,msg,type}."""
    if isinstance(exc, ValidationError):
        return [
            {
                "loc": list(item.get("loc", [])),
                "msg": item.get("msg", "validation error"),
                "type": item.get("type", "value_error"),
            }
            for item in exc.errors()
        ]

    if isinstance(exc, json.JSONDecodeError):
        return [{"loc": ["response"], "msg": exc.msg, "type": "json_decode_error"}]

    if isinstance(exc, ValueError):
        return [{"loc": ["response"], "msg": str(exc), "type": "value_error"}]

    return [{"loc": ["response"], "msg": str(exc), "type": exc.__class__.__name__}]

In [ ]:
# Happy-path run on good_source.md.
import yaml

good_source = Path("data/poc-wiki/raw/good_source.md")
if not good_source.exists():
    good_source.parent.mkdir(parents=True, exist_ok=True)
    good_source.write_text(
        """# Apollo launch sync\n\nSandro and Priya reviewed Apollo launch blockers.\nThe team agreed to keep weekly check-ins and track KPI drift.\n""",
        encoding="utf-8",
    )

content = good_source.read_text(encoding="utf-8")
analysis = await analyze_source(content, SourceKind.LOCAL_FILE)
print(analysis.model_dump_json(indent=2))

page, log = await synthesize_page(analysis, hint=None, existing_pages=[])
print(log)
assert page.status == PageStatus.ACTIVE
assert not page.validation_errors
assert page.type == PageType.SOURCE
assert page.sources
assert len(log) == 1

print(yaml.safe_dump(page.model_dump(mode="json"), sort_keys=False))

In [ ]:
# Retry-path run on ambiguous_source.md (expect recovery after validation feedback).
ambiguous_source = Path("data/poc-wiki/raw/ambiguous_source.md")
if not ambiguous_source.exists():
    ambiguous_source.parent.mkdir(parents=True, exist_ok=True)
    ambiguous_source.write_text(
        """# Apollo ambiguity note

The Apollo team wants an updated KPI rollout plan.
Apollo also appears in docs as a project codename, and references are mixed.
Sandro and Priya discussed this in a short sync.
""",
        encoding="utf-8",
    )

content = ambiguous_source.read_text(encoding="utf-8")
analysis = await analyze_source(content, SourceKind.LOCAL_FILE)
print(analysis.model_dump_json(indent=2))

# Intentionally nudge a likely schema error first (bare-name owners), so retry feedback can correct it.
retry_hint = "For frontmatter, include owners as bare names like Sandro and Priya."
page, log = await synthesize_page(analysis, hint=retry_hint, existing_pages=[])

assert len(log) >= 2
assert "validation_errors" in log[0]

print("Attempt 1 validation_errors:")
print(json.dumps(log[0]["validation_errors"], indent=2))
print("Final status:", page.status)
print("Attempts:", len(log))
print(log)

In [ ]:
# Draft-fallback run on garbage_source.md.
import yaml

garbage_source = Path("data/poc-wiki/raw/garbage_source.md")
if not garbage_source.exists():
    garbage_source.parent.mkdir(parents=True, exist_ok=True)
    garbage_source.write_text(
        """### ???
@@@ ??? ###
this is half notes / half noise / no stable structure
Sandro?? Apollo?? maybe, maybe not
<broken>tag soup without closure
""",
        encoding="utf-8",
    )

content = garbage_source.read_text(encoding="utf-8")
analysis = await analyze_source(content, SourceKind.LOCAL_FILE)

# Force retry exhaustion to demonstrate explicit draft fallback behavior.
_original_parser = _parse_frontmatter_and_body

def _always_fail_parser(raw: str) -> tuple[dict, str]:
    raise ValueError("forced parse failure for draft-fallback demo")

_parse_frontmatter_and_body = _always_fail_parser
try:
    page, log = await synthesize_page(analysis, hint=None, existing_pages=[])
finally:
    _parse_frontmatter_and_body = _original_parser

assert page.status == PageStatus.DRAFT
assert page.validation_errors
assert len(log) == MAX_ATTEMPTS

print("Attempt log:")
print(json.dumps(log, indent=2))

frontmatter_dict = page.model_dump(mode="json")
body_md = """# Needs Human Attention

This page failed strict schema synthesis after max retries.
Review the source content and validation errors before merging.

## Raw Source Excerpt

```text
{content}
```
""".format(content=content[:1200])

print("\nRendered draft page:\n")
print("---")
print(yaml.safe_dump(frontmatter_dict, sort_keys=False).strip())
print("---")
print(body_md)

In [ ]:
# Cost + token summary across good / ambiguous / garbage fixtures.
# as of 2026-04-30 — re-verify before re-running.
import hashlib
import time

from rich.console import Console
from rich.table import Table

PRICES = {
    "claude-haiku-4-5": {"input_per_million": 1.00, "output_per_million": 5.00},
    "claude-sonnet-4-6": {"input_per_million": 3.00, "output_per_million": 15.00},
}


def _estimate_cost_usd(ai: int, ao: int, si: int, so: int) -> float:
    haiku = PRICES["claude-haiku-4-5"]
    sonnet = PRICES["claude-sonnet-4-6"]
    return (
        (ai / 1_000_000) * haiku["input_per_million"]
        + (ao / 1_000_000) * haiku["output_per_million"]
        + (si / 1_000_000) * sonnet["input_per_million"]
        + (so / 1_000_000) * sonnet["output_per_million"]
    )


async def _analyze_with_usage(content: str, source_kind: SourceKind) -> tuple[SourceAnalysis, int, int]:
    sha = hashlib.sha256(content.encode()).hexdigest()
    user_msg = ANALYZE_USER_TEMPLATE.format(
        source_kind=source_kind.value,
        sha=sha,
        content=content,
        schema_json=json.dumps(SourceAnalysis.model_json_schema()),
    )
    purpose = getattr(config, "purpose", None) or config.purpose_body
    agents_style = getattr(config, "agents_style", None) or config.agents_body
    system = (
        ANALYZE_SYSTEM
        + f"\n\n<wiki_purpose>\n{purpose}\n</wiki_purpose>"
        + f"\n\n<style_guide>\n{agents_style}\n</style_guide>"
    )

    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        temperature=0,
        system=system,
        messages=[{"role": "user", "content": user_msg}],
    )
    data = _parse_json_object(resp.content[0].text)
    data["content_sha256"] = sha
    analysis_obj = SourceAnalysis.model_validate(data)
    return analysis_obj, resp.usage.input_tokens, resp.usage.output_tokens


async def _run_fixture(name: str, path: Path, hint: str | None = None, force_fallback: bool = False) -> dict:
    content = path.read_text(encoding="utf-8")
    t0 = time.monotonic()
    analysis_obj, analyze_in, analyze_out = await _analyze_with_usage(content, SourceKind.LOCAL_FILE)

    if force_fallback:
        original_parser = _parse_frontmatter_and_body

        def _fail_parser(raw: str) -> tuple[dict, str]:
            raise ValueError("forced parse failure for cost-table fixture")

        globals()["_parse_frontmatter_and_body"] = _fail_parser
        try:
            _, synth_log = await synthesize_page(analysis_obj, hint=hint, existing_pages=[])
        finally:
            globals()["_parse_frontmatter_and_body"] = original_parser
    else:
        _, synth_log = await synthesize_page(analysis_obj, hint=hint, existing_pages=[])

    synth_in = sum(item.get("input_tokens", 0) for item in synth_log)
    synth_out = sum(item.get("output_tokens", 0) for item in synth_log)
    wall_s = time.monotonic() - t0

    return {
        "fixture": name,
        "analyze_in": analyze_in,
        "analyze_out": analyze_out,
        "synth_attempts": len(synth_log),
        "synth_in": synth_in,
        "synth_out": synth_out,
        "cost_usd": _estimate_cost_usd(analyze_in, analyze_out, synth_in, synth_out),
        "wall_s": wall_s,
    }


good_metrics = await _run_fixture(
    "good_source.md",
    Path("data/poc-wiki/raw/good_source.md"),
    hint=None,
)
ambiguous_metrics = await _run_fixture(
    "ambiguous_source.md",
    Path("data/poc-wiki/raw/ambiguous_source.md"),
    hint="For frontmatter, include owners as bare names like Sandro and Priya.",
)
garbage_metrics = await _run_fixture(
    "garbage_source.md",
    Path("data/poc-wiki/raw/garbage_source.md"),
    hint=None,
    force_fallback=True,
)

rows = [good_metrics, ambiguous_metrics, garbage_metrics]

console = Console()
table = Table(title="Ingest Cost/Token Summary")
table.add_column("fixture")
table.add_column("analyze tokens (in/out)", justify="right")
table.add_column("synth attempts", justify="right")
table.add_column("synth tokens (in/out)", justify="right")
table.add_column("$ estimate", justify="right")
table.add_column("wall time", justify="right")

for r in rows:
    table.add_row(
        r["fixture"],
        f"{r['analyze_in']}/{r['analyze_out']}",
        str(r["synth_attempts"]),
        f"{r['synth_in']}/{r['synth_out']}",
        f"${r['cost_usd']:.4f}",
        f"{r['wall_s']:.2f}s",
    )

console.print(table)

In [ ]:
# Cell 17 — A/B prompt comparison (v1 vs v2) on ambiguous_source.md.
from statistics import mean

from rich.table import Table


async def synthesize_page_v2(
    analysis: SourceAnalysis,
    hint: str | None,
    existing_pages: list[dict],
) -> tuple[SourcePage, list[dict]]:
    """Variant with stronger instruction when constraints cannot be satisfied."""
    attempt_log: list[dict] = []
    prior_errors: list[dict] = []

    for attempt in range(1, MAX_ATTEMPTS + 1):
        user_msg = SYNTH_USER_TEMPLATE.format(
            analysis_json=analysis.model_dump_json(),
            existing_pages_json=json.dumps(existing_pages),
            schema_json=json.dumps(SourcePage.model_json_schema()),
        )
        user_msg += (
            "\n\n<constraint_handling>"
            "If you cannot satisfy a constraint, state which constraint and why, "
            "then still return best-effort <frontmatter>/<body> output for validation feedback."
            "</constraint_handling>"
        )

        if hint:
            user_msg += f"\n\n<hint>\n{hint}\n</hint>"
        if prior_errors:
            user_msg += (
                "\n\n<validation_errors>\n"
                + json.dumps(prior_errors, indent=2)
                + "\n</validation_errors>"
            )

        t0 = time.monotonic()
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=2048,
            temperature=0,
            system=SYNTH_SYSTEM + scope_blocks,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = resp.content[0].text
        attempt_log.append(
            {
                "attempt": attempt,
                "ms": int((time.monotonic() - t0) * 1000),
                "input_tokens": resp.usage.input_tokens,
                "output_tokens": resp.usage.output_tokens,
            }
        )

        try:
            fm_dict, body = _parse_frontmatter_and_body(raw)
            _ = body
            page = SourcePage.model_validate(fm_dict)
            return page, attempt_log
        except (ValidationError, ValueError, json.JSONDecodeError) as exc:
            prior_errors = _normalize_errors(exc)
            attempt_log[-1]["validation_errors"] = prior_errors

    draft = SourcePage.model_construct(
        title=analysis.proposed_title or "[unknown]",
        type=PageType.SOURCE,
        status=PageStatus.DRAFT,
        created=date.today(),
        last_synced=date.today(),
        sources=[],
        validation_errors=[json.dumps(err) for err in prior_errors],
    )
    return draft, attempt_log


async def _run_trials(label: str, synth_fn, analysis_obj: SourceAnalysis, trials: int = 5) -> dict:
    attempts: list[int] = []
    total_tokens: list[int] = []
    wall_s: list[float] = []

    for _ in range(trials):
        t0 = time.monotonic()
        _, log = await synth_fn(analysis_obj, hint=None, existing_pages=[])
        wall_s.append(time.monotonic() - t0)
        attempts.append(len(log))
        total_tokens.append(
            sum(item.get("input_tokens", 0) + item.get("output_tokens", 0) for item in log)
        )

    return {
        "label": label,
        "mean_attempts": mean(attempts),
        "mean_total_tokens": mean(total_tokens),
        "mean_wall_s": mean(wall_s),
    }


ab_content = Path("data/poc-wiki/raw/ambiguous_source.md").read_text(encoding="utf-8")
ab_analysis = await analyze_source(ab_content, SourceKind.LOCAL_FILE)

v1_metrics = await _run_trials("v1", synthesize_page, ab_analysis, trials=5)
v2_metrics = await _run_trials("v2", synthesize_page_v2, ab_analysis, trials=5)

ab_table = Table(title="A/B Prompt Comparison (ambiguous_source.md, 5 trials)")
ab_table.add_column("prompt")
ab_table.add_column("mean attempts to success", justify="right")
ab_table.add_column("mean total tokens", justify="right")
ab_table.add_column("mean wall time", justify="right")

for m in [v1_metrics, v2_metrics]:
    ab_table.add_row(
        m["label"],
        f"{m['mean_attempts']:.2f}",
        f"{m['mean_total_tokens']:.0f}",
        f"{m['mean_wall_s']:.2f}s",
    )

# Winner policy: fewer attempts, then fewer tokens, then lower wall time.
def _winner(a: dict, b: dict) -> str:
    key_a = (a["mean_attempts"], a["mean_total_tokens"], a["mean_wall_s"])
    key_b = (b["mean_attempts"], b["mean_total_tokens"], b["mean_wall_s"])
    return a["label"] if key_a <= key_b else b["label"]

winner = _winner(v1_metrics, v2_metrics)
console.print(ab_table)
print("Winner:", winner)

if winner == "v2":
    PROMPT_VERSION = "v2"

# when this prompt is extracted, bumping PROMPT_VERSION must coincide with a CACHE_VERSION bump in the same PR (CLAUDE.md, design §7.6).

## A/B Decision

A/B on `ambiguous_source.md` (5 trials each) selected **v1**.

- Mean attempts to success: tie (v1 = 1.00, v2 = 1.00)
- Mean total tokens: v1 lower (2195 vs 2224)
- Mean wall time: v1 lower (10.26s vs 11.12s)

Decision: keep current synth prompt as-is and **do not** adopt v2.

`PROMPT_VERSION` remains `v1`. If a future run adopts v2, bump `PROMPT_VERSION` in lockstep with `CACHE_VERSION` in the same PR (CLAUDE.md, design §7.6).

## Closing Notes

This notebook ran three fixture paths end-to-end. `good_source.md` succeeded on the happy path in one synth attempt, `ambiguous_source.md` demonstrated useful retry recovery (first-attempt validation errors fed back, then success), and `garbage_source.md` demonstrated the 3-attempt cap and explicit draft fallback with `validation_errors` for human attention.

Several pieces are still intentionally hand-waved at this stage: no real source adapter (fixtures are local files), no orchestrator/subagent routing, no cache layer implementation, and synthesis targets only `SourcePage` (not the broader cross-page or multi-type flows).

Handoff to Notebook 03: reuse `analyze_source` and `synthesize_page` from this notebook as the fixed harness, then compare Haiku vs Sonnet vs Opus on the same fixtures (quality, retries, tokens, latency, and cost) to produce model-selection receipts.